# Lesson 5: Jev Use Cases in Code

One worked example for each category from Lesson 04, run on real data:

1. **AI-powered workflows:** route support tickets; escalate the unsure ones to an LLM.
2. **Map-reducing big data:** label every commit in a public repo's git history.
3. **Real-time applications:** Jev plays text adventure games, one action per turn.
4. **Verify everything:** check scientific claims against the abstracts they cite.

Set up the repo first by following **Setup** in the root `README.md`. Every section needs `TYPESAFE_API_KEY`. Section 1's escalation step also needs `OPENAI_API_KEY`.

The data is downloaded or generated when you run the notebook, into `data/` next to this notebook and the Hugging Face cache. Costs use each API's reported token usage at standard uncached USD rates, verified 2026-09-30. Latency is wall-clock time from your machine, so it includes the network round trip.

## Setup

`ask_all` sends the same questions about many inputs, `CONCURRENCY` calls at a time. Jev has no batch endpoint, so this is how you map over a dataset.

TypeSafe publishes rate limits of 80 requests/s and 100K tokens/s ([docs.typesafe.ai/models](https://docs.typesafe.ai/models), checked 2026-10-06). At ~150 ms per call, 12 concurrent calls is about 80 requests/s. The SDK retries `429` responses, honoring `Retry-After`; a `TypeSafeRateLimitError` that survives the retries is raised.

In [ ]:
import asyncio
import statistics
import time
from pathlib import Path

import pandas as pd
from typesafe_sdk import AsyncTypeSafeClient, Choice, Noul, RetryPolicy, TypeSafeClient

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

CONCURRENCY = 12
JEV_USD_PER_INPUT_TOKEN = 0.042 / 1_000_000  # outputs are free


def jev_cost(responses) -> float:
    return sum(r.usage.input_tokens for r in responses) * JEV_USD_PER_INPUT_TOKEN


async def ask_all(states, questions):
    """Ask the same questions about every state. Returns (responses, latencies in ms)."""
    semaphore = asyncio.Semaphore(CONCURRENCY)

    async with AsyncTypeSafeClient(retry=RetryPolicy(max_retries=5)) as client:
        async def ask(state):
            async with semaphore:
                start = time.perf_counter()
                response = await client.system_one(state=state, questions=questions)
                return response, (time.perf_counter() - start) * 1000

        start = time.perf_counter()
        results = await asyncio.gather(*(ask(state) for state in states))
        wall_s = time.perf_counter() - start

    responses = [response for response, _ in results]
    latencies = [ms for _, ms in results]
    print(f"{len(states):,} Jev calls in {wall_s:.1f} s | "
          f"median {statistics.median(latencies):.0f} ms per call | "
          f"{sum(r.usage.input_tokens for r in responses):,} input tokens | ${jev_cost(responses):.4f}")
    return responses, latencies


def threshold_table(df, correct, confidence, thresholds=(0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99)):
    """For each confidence threshold: how much is handled automatically, and how accurately."""
    rows = []
    for threshold in thresholds:
        kept = df[df[confidence] >= threshold]
        rows.append({
            "threshold": threshold,
            "handled automatically": f"{len(kept) / len(df):.0%}",
            "accuracy": f"{kept[correct].mean():.1%}",
            "errors": int((~kept[correct]).sum()),
        })
    return pd.DataFrame(rows)

## 1. AI-Powered Workflows: Route Support Tickets

Modeled on the support-triage projects in Lesson 04. Jev picks a queue for each ticket and flags status requests. Code routes the confident ones and escalates the rest to GPT-6 Luna.

**Data:** the [Bitext customer support dataset](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) (CDLA-Sharing-1.0): 26,872 customer messages, each labeled with one of 11 categories and 27 intents. The labels let us measure routing accuracy.

In [ ]:
from datasets import load_dataset

N_TICKETS = 500

tickets = (
    load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
    .shuffle(seed=0)
    .select(range(N_TICKETS))
    .to_pandas()[["instruction", "category", "intent"]]
)
tickets.head()

### The Jev call: one `Choice` and one `Noul` per ticket

The queues are the dataset's 11 categories. Each description summarizes the intents in that category.

In [ ]:
QUEUES = {
    "ACCOUNT": "Create, edit, switch, or delete an account; registration and password problems",
    "CANCEL": "Cancellation fees and penalties",
    "CONTACT": "Reach customer service or a human agent",
    "DELIVERY": "Delivery options and how long delivery takes",
    "FEEDBACK": "Complaints, claims, and reviews",
    "INVOICE": "Find or check an invoice",
    "ORDER": "Place, change, cancel, or track an order",
    "PAYMENT": "Payment methods and payment problems",
    "REFUND": "Refund policy, request a refund, or track a refund",
    "SHIPPING": "Set up or change a shipping address",
    "SUBSCRIPTION": "Newsletter subscription",
}

TICKET_QUESTIONS = {
    "queue": Choice(
        instructions="Which support queue should handle this message?",
        criteria=QUEUES,
    ),
    "status_request": Noul(
        instructions="Is the customer asking for the status of an order or refund they already have?",
        criteria={
            "true": "Wants to track an order, a shipment, or a refund already requested",
            "false": "Anything else: new requests, changes, cancellations, policies, fees, complaints",
        },
    ),
}

### The code that acts on the answers

Branch order matters: the confidence check runs first, so an unsure ticket never gets an automatic reply.

In [ ]:
AUTO_ROUTE = 0.90  # queue confidence needed to act without an LLM
AUTO_REPLY = 0.90  # status_request probability needed to send a tracking link


def route(answers) -> str:
    queue = answers["queue"]
    if queue.confidence < AUTO_ROUTE:
        return "escalate"                      # unsure: an LLM reads it
    if answers["status_request"].noul >= AUTO_REPLY:
        return f"auto_reply:{queue.choice}"    # send the order or refund tracking link
    return f"queue:{queue.choice}"             # confident: route it

In [ ]:
responses, latencies = await ask_all(tickets["instruction"].tolist(), TICKET_QUESTIONS)

tickets["queue"] = [r.answers["queue"].choice for r in responses]
tickets["confidence"] = [r.answers["queue"].confidence for r in responses]
tickets["status_request"] = [r.answers["status_request"].noul for r in responses]
tickets["action"] = [route(r.answers) for r in responses]
tickets["correct"] = tickets["queue"] == tickets["category"]
ticket_jev_cost = jev_cost(responses)

tickets["action"].str.split(":").str[0].value_counts()

In [ ]:
handled = tickets[tickets["action"] != "escalate"]
replies = tickets[tickets["action"].str.startswith("auto_reply")]
is_status = tickets["intent"].isin(["track_order", "track_refund"])

print(f"Routed without a human: {len(handled) / len(tickets):.0%} of tickets, "
      f"{handled['correct'].mean():.1%} to the right queue")
print(f"Automatic replies: {len(replies)}, "
      f"{replies['intent'].isin(['track_order', 'track_refund']).mean():.0%} were status requests")
print(f"Status requests in the sample: {is_status.sum()}, "
      f"{replies.index.isin(tickets[is_status].index).sum()} answered automatically")
print(f"Accuracy if we trusted every answer: {tickets['correct'].mean():.1%}")

### Calibration pays off

Raise the threshold and fewer tickets are handled automatically, but the ones that are get more accurate. That trade-off only works because the confidence means something (RLCD, Lesson 02). Pick the threshold from your own labeled data.

In [ ]:
threshold_table(tickets, correct="correct", confidence="confidence")

In [ ]:
pd.set_option("display.max_colwidth", 90)
tickets[~tickets["correct"]].sort_values("confidence")[["instruction", "category", "intent", "queue", "confidence"]]

### Escalate the unsure tickets to an LLM

GPT-6 Luna reads each escalated ticket and picks a queue: one structured-output call per ticket, with reasoning effort `none`. Luna is OpenAI's cheapest GPT-6 model.

In [ ]:
from typing import Literal

import openai
from pydantic import BaseModel

LUNA_USD_PER_TOKEN = {"input": 0.10 / 1_000_000, "output": 0.50 / 1_000_000}  # verified 2026-10-06


class LLMRouting(BaseModel):
    queue: Literal[tuple(QUEUES)]  # one of the QUEUES keys


llm = openai.AsyncOpenAI()  # reads OPENAI_API_KEY
llm_semaphore = asyncio.Semaphore(8)


async def route_with_llm(ticket: str) -> dict:
    async with llm_semaphore:
        start = time.perf_counter()
        response = await llm.responses.parse(
            model="gpt-6-luna",
            input=f"Route this support message to one queue.\n\nQueues: {QUEUES}\n\nMessage: {ticket}",
            text_format=LLMRouting,
            reasoning={"effort": "none"},
        )
        latency_ms = (time.perf_counter() - start) * 1000
    if response.status != "completed":
        raise RuntimeError(f"GPT-6 Luna stopped with status {response.status}")
    usage = response.usage
    return {
        "llm_queue": response.output_parsed.queue,
        "llm_ms": latency_ms,
        "llm_usd": usage.input_tokens * LUNA_USD_PER_TOKEN["input"]
        + usage.output_tokens * LUNA_USD_PER_TOKEN["output"],
    }


escalated = tickets[tickets["action"] == "escalate"].copy()
llm_results = await asyncio.gather(*(route_with_llm(t) for t in escalated["instruction"]))
escalated = escalated.join(pd.DataFrame(llm_results, index=escalated.index))

print(f"{len(escalated)} escalated tickets | GPT-6 Luna median {escalated['llm_ms'].median():,.0f} ms "
      f"| ${escalated['llm_usd'].sum():.4f}")
print(f"On these tickets: Jev's top choice {escalated['correct'].mean():.0%} right, "
      f"GPT-6 Luna {(escalated['llm_queue'] == escalated['category']).mean():.0%} right")

### The whole pipeline vs one LLM call per ticket

The baseline sends every ticket to GPT-6 Luna: one call each. Its cost and latency per ticket are the averages measured on the escalated tickets above.

In [ ]:
pipeline_correct = handled["correct"].sum() + (escalated["llm_queue"] == escalated["category"]).sum()
pipeline_cost = ticket_jev_cost + escalated["llm_usd"].sum()

print(f"Pipeline accuracy: {pipeline_correct / len(tickets):.1%} "
      f"({len(handled)} routed by Jev, {len(escalated)} by GPT-6 Luna)")
print("Cost per 1,000 tickets:")
print(f"  Jev on every ticket + GPT-6 Luna on escalations: ${pipeline_cost / len(tickets) * 1000:.3f}")
print(f"  GPT-6 Luna on every ticket (estimated):          ${escalated['llm_usd'].mean() * 1000:.3f}")
print("Median latency per ticket:")
print(f"  Jev: {statistics.median(latencies):,.0f} ms | GPT-6 Luna: {escalated['llm_ms'].median():,.0f} ms")

## 2. Map-Reducing Big Data: Label a Repo's Git History

Modeled on [commit-miner](https://github.com/devanshbatham/commit-miner). Map: ask Jev the same questions about every commit. Reduce: count, group, and filter the answers with pandas.

**Data:** the full git history of [`psf/requests`](https://github.com/psf/requests) (Apache-2.0). A blobless clone downloads commits and file trees but no file contents, so it takes about a second. Each commit becomes its message plus the paths it changed. commit-miner also reads the diffs; we skip them to keep the clone small.

In [ ]:
import subprocess

REPO_URL = "https://github.com/psf/requests.git"
REPO_DIR = DATA_DIR / "requests.git"
MAX_COMMITS = 5000  # requests has ~4,900 non-merge commits

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", "--bare", "--filter=blob:none", REPO_URL, str(REPO_DIR)], check=True)

log = subprocess.run(
    ["git", "-C", str(REPO_DIR), "log", "--no-merges", f"--max-count={MAX_COMMITS}",
     "--date=short", "--format=%x00%h%x1f%ad%x1f%B%x1e", "--name-only"],
    capture_output=True, text=True, check=True,
).stdout

commits = []
for record in log.split("\x00")[1:]:
    header, files = record.split("\x1e")
    sha, date, message = header.split("\x1f")
    commits.append({"sha": sha, "date": date, "message": message.strip(), "files": files.strip().splitlines()})

commits = pd.DataFrame(commits)
print(f"{len(commits):,} commits from {commits['date'].min()} to {commits['date'].max()}")
commits.head()

### The Jev call: what kind of change, and is it a security fix?

A `Choice` gives each commit one kind, so we can group by it. commit-miner uses independent `Noul`s instead, because one commit can be several things at once.

In [ ]:
COMMIT_QUESTIONS = {
    "kind": Choice(
        instructions="What kind of change is this commit? Judge `message` and the changed `files`.",
        criteria={
            "bug_fix": "Fixes incorrect behavior",
            "feature": "Adds new behavior or options",
            "refactor": "Restructures or cleans up code without changing behavior",
            "docs": "Documentation only",
            "tests": "Tests only",
            "build": "Dependencies, CI, packaging, or releases",
        },
    ),
    "security_fix": Noul(
        instructions=(
            "Does this commit fix or harden a security weakness "
            "that could affect confidentiality, integrity, or availability?"
        ),
        criteria={
            "true": "Fixes a vulnerability or strengthens a security control",
            "false": "An ordinary bug fix, feature, refactor, docs, tests, or build change",
        },
    ),
}

### Map

In [ ]:
states = [{"message": c.message, "files": c.files} for c in commits.itertuples()]
responses, _ = await ask_all(states, COMMIT_QUESTIONS)

commits["kind"] = [r.answers["kind"].choice for r in responses]
commits["security_fix"] = [r.answers["security_fix"].noul for r in responses]
commits["year"] = commits["date"].str[:4]

### Reduce: what kind of work happened each year?

In [ ]:
pd.crosstab(commits["year"], commits["kind"], margins=True)

### Reduce: find the security fixes

A keyword search finds commits that *mention* security. Jev judges what the commit *does*.

In [ ]:
SECURITY_AT = 0.8
KEYWORDS = r"secur|CVE|vulnerab|leak"

commits["first_line"] = commits["message"].str.split("\n").str[0]
commits["keyword_match"] = commits["message"].str.contains(KEYWORDS, case=False)
flagged = commits[commits["security_fix"] >= SECURITY_AT].sort_values("security_fix", ascending=False)

print(f"Jev flags {len(flagged)} commits as security fixes (probability >= {SECURITY_AT}); "
      f"{(~flagged['keyword_match']).sum()} of them never match /{KEYWORDS}/")
flagged[["sha", "date", "security_fix", "keyword_match", "first_line"]]

In [ ]:
# The other direction: keyword matches that Jev did not flag
commits[commits["keyword_match"] & (commits["security_fix"] < SECURITY_AT)].sort_values(
    "security_fix", ascending=False
)[["sha", "date", "security_fix", "first_line"]].head(15)

## 3. Real-Time Applications: Jev Plays Text Adventure Games

Modeled on [minecraft-agent](https://github.com/rmalde/minecraft-agent) and [jev-ultrafast](https://github.com/browser-use/jev-ultrafast): every turn, the environment lists the actions available, Jev picks one as a `Choice`, and code executes it.

**Data:** [TextWorld](https://github.com/microsoft/TextWorld) (Microsoft Research, MIT license) cooking games, like those in the First TextWorld Problems competition. Find the cookbook, collect the ingredients, cut and cook them as the recipe says, then prepare and eat the meal. The games are generated on your machine.

There's no planner model here. Once Jev reads the cookbook, code keeps the recipe in `state` for every later turn. Without that, Jev picks plausible actions with no plan, like eating a raw ingredient.

In [ ]:
import textworld
import textworld.challenges

N_GAMES = 10
MAX_STEPS = 50
LATENCY_BUDGET_MS = 300  # per decision
GAME_DIR = DATA_DIR / "textworld"
GAME_DIR.mkdir(exist_ok=True)

_, make_cooking_game, _ = textworld.challenges.CHALLENGES["tw-cooking"]


def build_game(seed: int) -> Path:
    """A cooking game: 2 ingredients to find across 6 rooms, with doors, cutting, and cooking."""
    path = GAME_DIR / f"cooking-{seed}.z8"
    if not path.exists():
        options = textworld.GameOptions()
        options.seeds = seed
        options.path = str(path)
        settings = {"recipe": 2, "take": 2, "go": 6, "open": True, "cook": True, "cut": True,
                    "drop": False, "recipe_seed": seed, "split": None}
        textworld.generator.compile_game(make_cooking_game(settings, options), options)
    return path


games = [build_game(seed) for seed in range(1, N_GAMES + 1)]

### The Jev call: pick the next command

The game's admissible commands become the `Choice` options, so Jev can only pick a command the game accepts.

Two details matter a lot. `clean` strips the game's status bar from each observation, and the instructions carry three rules for the slips Jev made without them: taking the wrong ingredient, and cooking one twice (which burns it and loses the game). When I ran these 10 games on 2026-10-06, a one-line instruction without the rules won 4; this version won 9. Your runs can differ.

In [ ]:
INFOS = textworld.EnvInfos(admissible_commands=True, objective=True, inventory=True, max_score=True)
client = TypeSafeClient()

ACTION_INSTRUCTIONS = (
    "Which command makes the most progress toward `goal`? "
    "Once `recipe` has been read, follow its directions in order. "
    "Take only the exact ingredients `recipe` lists; open containers to find missing ones. "
    "Never cook or cut the same ingredient twice."
)


def clean(text: str) -> str:
    """Drop the game's prompt and status bar, which follow the last '>'."""
    return text.split("\n>")[0].strip()


def next_action(game_state, recipe: str | None, history: list[str]):
    commands = [c for c in game_state.admissible_commands if c not in ("look", "inventory")]
    state = {
        "goal": game_state.objective,
        "recipe": recipe or "not read yet",
        "observation": clean(game_state.feedback),
        "inventory": game_state.inventory.strip(),
        "recent_actions": history[-8:],
    }
    start = time.perf_counter()
    response = client.system_one(state=state, questions={"action": Choice(
        instructions=ACTION_INSTRUCTIONS,
        criteria={command: None for command in commands},
    )})
    latency_ms = (time.perf_counter() - start) * 1000
    return response.answers["action"], latency_ms, response

### The loop: observe, decide, act

In [ ]:
def play(path: Path, verbose: bool = False) -> dict:
    env = textworld.start(str(path), request_infos=INFOS)
    game_state = env.reset()
    recipe, history, latencies, responses = None, [], [], []
    if verbose:
        print(f"Objective: {game_state.objective}\n")

    for step in range(1, MAX_STEPS + 1):
        action, latency_ms, response = next_action(game_state, recipe, history)
        game_state, score, done = env.step(action.choice)
        if action.choice == "examine cookbook":
            recipe = clean(game_state.feedback)  # code keeps the plan
        history.append(action.choice)
        latencies.append(latency_ms)
        responses.append(response)
        if verbose:
            print(f"{step:2d}  {latency_ms:4.0f} ms  confidence {action.confidence:.2f}  "
                  f"{action.choice:<40} score {score}/{game_state.max_score}")
        if done:
            break
    env.close()

    return {
        "game": path.stem,
        "won": game_state.won,
        "lost": game_state.lost,
        "steps": step,
        "score": f"{score}/{game_state.max_score}",
        "median ms": round(statistics.median(latencies)),
        "max ms": round(max(latencies)),
        "over budget": sum(ms > LATENCY_BUDGET_MS for ms in latencies),
        "cost": f"${jev_cost(responses):.5f}",
    }


results = [play(games[0], verbose=True)]

In [ ]:
results += [play(game) for game in games[1:]]
pd.DataFrame(results)

In [ ]:
runs = pd.DataFrame(results)
decisions = runs["steps"].sum()
print(f"Won {runs['won'].sum()} of {len(runs)} games in {decisions} decisions; "
      f"{runs['over budget'].sum()} decisions over the {LATENCY_BUDGET_MS} ms budget")

## 4. Verify Everything: Check Claims Against Evidence

Modeled on Jev MCP's [`jev_verify`](https://github.com/jkudish/jev-mcp#jev_verify) and TypeSafe's [citation-check cookbook](https://docs.typesafe.ai/cookbooks/citation_check). For each claim, Jev decides whether the evidence supports it, contradicts it, or says nothing about it. Confident verdicts stand; the rest go to review.

**Data:** [SciFact](https://github.com/allenai/scifact) via [`allenai/scifact_entailment`](https://huggingface.co/datasets/allenai/scifact_entailment), validation split: 340 scientific claims, each paired with a paper abstract and a label from expert annotators. Claims and labels are CC BY 4.0; abstracts are ODC-By 1.0.

In [ ]:
claims = load_dataset("allenai/scifact_entailment", revision="refs/convert/parquet", split="validation").to_pandas()
claims["evidence"] = claims["abstract"].apply(" ".join)
claims["gold"] = claims["verdict"].map({"SUPPORT": "supports", "CONTRADICT": "contradicts", "NEI": "says_nothing"})

print(claims["gold"].value_counts().to_string(), "\n")
claims[["claim", "title", "gold"]].head()

### The Jev call: one `Choice` per claim

In [ ]:
CLAIM_QUESTIONS = {
    "relation": Choice(
        instructions="How does `evidence` relate to `claim`?",
        criteria={
            "supports": "The evidence states the claim or directly implies that it is true",
            "contradicts": "The evidence states the opposite of the claim or implies that it is false",
            "says_nothing": "The evidence does not address what the claim asserts, either way",
        },
    ),
}

AUTO_ACCEPT = 0.8  # confidence needed for the verdict to stand without review

In [ ]:
states = [{"claim": c.claim, "evidence": c.evidence} for c in claims.itertuples()]
responses, _ = await ask_all(states, CLAIM_QUESTIONS)

claims["verdict_jev"] = [r.answers["relation"].choice for r in responses]
claims["confidence"] = [r.answers["relation"].confidence for r in responses]
claims["probability"] = [r.answers["relation"].probabilities[r.answers["relation"].choice] for r in responses]
claims["action"] = ["auto" if c >= AUTO_ACCEPT else "review" for c in claims["confidence"]]
claims["correct"] = claims["verdict_jev"] == claims["gold"]

auto = claims[claims["action"] == "auto"]
print(f"Verdicts that stand: {len(auto) / len(claims):.0%} of claims, {auto['correct'].mean():.1%} correct")
print(f"Sent to review: {(claims['action'] == 'review').sum()}")
print(f"Accuracy if we trusted every verdict: {claims['correct'].mean():.1%}")

In [ ]:
pd.crosstab(claims["gold"], claims["verdict_jev"], rownames=["gold"], colnames=["Jev"])

In [ ]:
threshold_table(claims, correct="correct", confidence="confidence")

### Check calibration on your own data

Group the verdicts by Jev's probability for its pick, then compare each group's average probability with how often it was right. If Jev were perfectly calibrated on this data, the two columns would match.

In [ ]:
claims["bucket"] = pd.cut(claims["probability"], [0, 0.6, 0.7, 0.8, 0.9, 1.0], include_lowest=True)
claims.groupby("bucket", observed=True).agg(
    verdicts=("correct", "size"),
    mean_probability=("probability", "mean"),
    accuracy=("correct", "mean"),
).round(2)